In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import sys

# load data set
df = pd.read_csv('../../NPFC-Test_Database_V2.csv')

In [7]:
import prepare_file

prepare_file.generate_prepared_file()

Preparing data file (this can take a few minutes)...


ValueError: Encountered all NA values

In [3]:
from data_preparation import align_lag_signals
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../../NPFC-Test_Database_V2.csv')
df = align_lag_signals(df, progress=True)

# 1. Selection
subjects = df['Subject_ID'].unique()[:3]

def time_to_seconds(t_str):
    try:
        h, m, s = map(int, t_str.split(':'))
    except:
        m, s = map(int, t_str.split(':'))
        h = 0
    return h * 3600 + m * 60 + s

def time_since_timestamp(timestamp):
    return lambda seconds: seconds - timestamp

def plot_task_data(task_id):

    df['Total_Seconds'] = df['Test_Time'].apply(time_to_seconds)

    fig, axes = plt.subplots(3, 1, figsize=(15, 14), sharex=False)

    for i, sub_id in enumerate(subjects):
        # Filter data
        sub_df = df[df['Subject_ID'] == sub_id]
        
        # Identify stimulus start (Test_Time)
        try:
            stim_time = sub_df[sub_df['Task_Num'] == task_id]['Total_Seconds'].iloc[0]
            # Get window: 5s before, 15s after
            plot_data = sub_df[(sub_df['Total_Seconds'] >= stim_time - 5) & 
                            (sub_df['Total_Seconds'] <= stim_time + 15)].copy()
            
            plot_data['Time_Since_Task'] = df["Total_Seconds"].apply(time_since_timestamp(stim_time))
            
            # 2. Normalize signals (0-1) for visual comparison
            for col in ['EDA_aligned', 'Alpha_TP9', 'Alpha_AF7', 'Alpha_AF8', 'Alpha_TP10', 'resmasknet_happiness_aligned']:
                plot_data[col] = (plot_data[col] - plot_data[col].min()) / (plot_data[col].max() - plot_data[col].min())
            
            # 3. Plotting
            ax = axes[i]
            ax.plot(plot_data['Time_Since_Task'], plot_data['EDA_aligned'], label='EDA (Slow)', color='blue', linewidth=2)
            ax.plot(plot_data['Time_Since_Task'], plot_data[['Alpha_TP9', 'Alpha_AF7', 'Alpha_AF8', 'Alpha_TP10']].mean(axis=1), label='Alpha EEG (Fast)', color='red', alpha=0.7)
            ax.plot(plot_data['Time_Since_Task'], plot_data['resmasknet_happiness_aligned'], label='Facial Happiness (Mid)', color='green', linestyle='--')
            
            ax.axvline(x=0, color='black', linestyle=':', label='Stimulus Onset')
            ax.set_title(f'Subject {sub_id}')
            ax.legend()
            ax.set_xlabel("Time (s) since stimulus onset")
            ax.set_ylabel("Normalized signal value (0-1)")
            
        except IndexError:
            print(f"Task {task_id} not found for Subject {sub_id}")

    plt.suptitle(f"Response Asynchrony during transition to task {task_id}")
    plt.tight_layout(pad=2)
    plt.show()

plot_task_data(3.1) # Math task
plot_task_data(6.1) # Video task

KeyboardInterrupt: 